In [9]:
# Select the venv kernel in VS Code (Python: env/bin/python)
import sys
print(sys.executable)

/Users/dharani/Desktop/PSI/env/bin/python


In [10]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd

papers = pd.read_csv(
    'Papers.csv'
)

papers.head()

,Unnamed: 0,title,authors,nc_state_people,DOI,PMID,year,url,topics,abstract
0,0,Mapping variability of soil water content and ...,"Sayde, Chadi; Buelga, Javier Benitez; Rodrigue...",Chadi Sayde (csayde),10.1002/2013wr014983,NaN,2014,https://openalex.org/W2160181936,Soil Moisture and Remote Sensing; Thermal Anal...,Abstract The Actively Heated Fiber Optic (AHFO...
1,1,Natural gas price uncertainty and the cost‐eff...,"Kern, Jordan D.; Characklis, Gregory W.; Foste...",Jordan Kern (jkern),10.1002/2014wr016533,NaN,2015,https://openalex.org/W2093917128,Water resources management and optimization; A...,Abstract Prolonged periods of low reservoir in...
2,2,High-resolution wind speed measurements using ...,"Sayde, Chadi; Thomas, Christoph K.; Wagner, Ja...",Chadi Sayde (csayde),10.1002/2015gl066729,NaN,2015,https://openalex.org/W2258664150,Seismic Waves and Analysis; Meteorological Phe...,Abstract We present a novel technique to simul...
3,3,Calibration of soil moisture sensing with subs...,"Benítez-Buelga, Javier; Rodríguez-Sinobas, Leo...",Chadi Sayde (csayde),10.1002/2015wr017897,NaN,2016,https://openalex.org/W2298265415,Soil Moisture and Remote Sensing; Soil and Uns...,Abstract The heat pulse probe method can be im...
4,4,Mapping high-resolution soil moisture and prop...,NaN,Chadi Sayde (csayde),10.1002/2016wr019031,NaN,2016,https://openalex.org/W2521661497,Soil Moisture and Remote Sensing; Soil and Uns...,This study demonstrated a new method for mappi...


In [12]:
papers = papers.drop(columns=['Unnamed: 0','PMID', 'url', 'abstract'])

In [13]:
papers = papers.dropna()

In [14]:
papers.isna().sum()

title              0
authors            0
nc_state_people    0
DOI                0
year               0
topics             0
dtype: int64

In [15]:
papers.shape

(3159, 6)

In [16]:
authors = papers['authors'].str.split(';')

In [17]:
papers['co_author'] = authors.str[1:].apply(
    lambda x: '; '.join([i.strip() for i in x]) if x else None
)

In [18]:
nc_authors_list = papers['nc_state_people'].str.split(';').apply(
    lambda x: [item.strip() for item in x] if x is not None else None
)

In [19]:
nc_authors_list.isna().sum()

np.int64(0)

In [20]:
import unicodedata
import re

ENTRY_PATTERN = re.compile(r"^\s*(.+?)\s*\((.*?)\)\s*$")


def normalize(text):
    text = unicodedata.normalize('NFKD', str(text))
    text = text.encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'[^a-z]', '', text.lower())


def split_name_tokens(name):
    return [normalize(t) for t in re.split(r"[\s,]+", str(name)) if normalize(t)]


def parse_nc_entry(entry):
    entry = str(entry).strip()
    if not entry:
        return None, None, True

    match = ENTRY_PATTERN.match(entry)
    if not match:
        # Keep the raw text as fallback name for author matching.
        return entry, None, True

    name = match.group(1).strip()
    unity_id = match.group(2).strip()

    # Empty unity IDs are invalid format for later filtering,
    # but we still keep the name for nc_authors matching.
    if not unity_id:
        return name, None, True

    return name, unity_id, False


def build_unity_aliases(papers_df):
    aliases = {}
    for value in papers_df['nc_state_people'].dropna():
        for entry in str(value).split(';'):
            name, unity_id, _ = parse_nc_entry(entry)
            if not name or not unity_id:
                continue
            aliases.setdefault(unity_id, set()).add(name)

    return {k: sorted(v) for k, v in aliases.items()}


def surname_candidates(name):
    tokens = split_name_tokens(name)
    if not tokens:
        return set()

    candidates = {tokens[-1]}
    if len(tokens) >= 2:
        candidates.add(tokens[-2])
        candidates.add(tokens[-2] + tokens[-1])
    if len(tokens) >= 3:
        candidates.add(tokens[-3] + tokens[-2])
        candidates.add(tokens[-3] + tokens[-2] + tokens[-1])

    return {c for c in candidates if len(c) >= 2}


def unity_candidates(unity_id):
    uid = normalize(unity_id)
    if not uid:
        return set()

    candidates = set()
    if len(uid) >= 5:
        candidates.add(uid)
        candidates.add(uid[1:])
    if len(uid) >= 6:
        candidates.add(uid[2:])

    return {c for c in candidates if len(c) >= 4}


def extract_nc_authors_and_ids(row):

    if pd.isna(row['nc_state_people']):
        return pd.Series([None, None])

    authors_list = [
        a.strip().rstrip(',')
        for a in str(row['authors']).split(';')
        if a.strip()
    ]

    nc_entries = [
        x.strip()
        for x in str(row['nc_state_people']).split(';')
        if x.strip()
    ]

    matched_authors = []
    unity_ids = []

    for entry in nc_entries:

        nc_name, unity_id, _ = parse_nc_entry(entry)
        if not nc_name:
            continue

        candidate_names = [nc_name]
        if unity_id and unity_id in unity_aliases:
            candidate_names.extend(unity_aliases[unity_id])

        seen = set()
        candidate_names = [n for n in candidate_names if not (n in seen or seen.add(n))]

        all_surname_candidates = set()
        for candidate_name in candidate_names:
            all_surname_candidates.update(surname_candidates(candidate_name))

        uid_candidates = unity_candidates(unity_id) if unity_id else set()

        selected_author = None
        for author in authors_list:
            author_norm = normalize(author)

            if any(c in author_norm for c in all_surname_candidates):
                selected_author = author
                break

            if uid_candidates and any(c in author_norm for c in uid_candidates):
                selected_author = author
                break

        if selected_author:
            matched_authors.append(selected_author)

        if unity_id:
            unity_ids.append(unity_id)

    return pd.Series([
        '; '.join(matched_authors) if matched_authors else None,
        '; '.join(unity_ids) if unity_ids else None
    ])


unity_aliases = build_unity_aliases(papers)

papers[['nc_authors', 'unity_ids']] = papers.apply(
    extract_nc_authors_and_ids,
    axis=1
)



In [21]:
papers.head()

,title,authors,nc_state_people,DOI,year,topics,co_author,nc_authors,unity_ids
0,Mapping variability of soil water content and ...,"Sayde, Chadi; Buelga, Javier Benitez; Rodrigue...",Chadi Sayde (csayde),10.1002/2013wr014983,2014,Soil Moisture and Remote Sensing; Thermal Anal...,"Buelga, Javier Benitez; Rodriguez-Sinobas, Leo...","Sayde, Chadi",csayde
1,Natural gas price uncertainty and the cost‐eff...,"Kern, Jordan D.; Characklis, Gregory W.; Foste...",Jordan Kern (jkern),10.1002/2014wr016533,2015,Water resources management and optimization; A...,"Characklis, Gregory W.; Foster, Benjamin T.","Kern, Jordan D.",jkern
2,High-resolution wind speed measurements using ...,"Sayde, Chadi; Thomas, Christoph K.; Wagner, Ja...",Chadi Sayde (csayde),10.1002/2015gl066729,2015,Seismic Waves and Analysis; Meteorological Phe...,"Thomas, Christoph K.; Wagner, James; Selker, John","Sayde, Chadi",csayde
3,Calibration of soil moisture sensing with subs...,"Benítez-Buelga, Javier; Rodríguez-Sinobas, Leo...",Chadi Sayde (csayde),10.1002/2015wr017897,2016,Soil Moisture and Remote Sensing; Soil and Uns...,"Rodríguez-Sinobas, Leonor; Sánchez Calvo, Raul...","Sayde, Chadi",csayde
5,Temporal variability in the importance of hydr...,"Nelson, Natalie G.; Muñoz-Carpena, Rafael; Nea...",Natalie Nelson (nnelson4),10.1002/2016WR020196,2017,Coastal wetland ecosystem dynamics; Marine and...,"Muñoz-Carpena, Rafael; Neale, Patrick J.; Tzor...","Nelson, Natalie G.",nnelson4


In [22]:
def count_items(value):
    if pd.isna(value) or value is None:
        return 0
    return len([x for x in str(value).split(';') if x.strip()])


def has_invalid_nc_state_people_format(value):
    if pd.isna(value) or value is None:
        return False

    for entry in str(value).split(';'):
        entry = entry.strip()
        if not entry:
            continue

        name, unity_id, is_invalid = parse_nc_entry(entry)
        if is_invalid:
            return True

    return False

papers['count_nc_state_people'] = papers['nc_state_people'].apply(count_items)
papers['count_nc_authors'] = papers['nc_authors'].apply(count_items)
papers['count_unity_ids'] = papers['unity_ids'].apply(count_items)

papers['has_invalid_nc_state_people_format'] = papers['nc_state_people'].apply(has_invalid_nc_state_people_format)

papers['counts_match'] = (
    (papers['count_nc_state_people'] == papers['count_nc_authors']) &
    (papers['count_nc_state_people'] == papers['count_unity_ids'])
)

mismatch_rows = papers[(~papers['has_invalid_nc_state_people_format']) & (papers['counts_match'] == False)]
invalid_format_rows = papers[papers['has_invalid_nc_state_people_format']]

if len(mismatch_rows) > 0:
    print(f"❌ MISMATCH FOUND (excluding invalid nc_state_people format rows) : { len(mismatch_rows)}")
    display(mismatch_rows[
        ['title',
         'authors',
         'nc_state_people',
         'nc_authors',
         'unity_ids',
         'count_nc_state_people',
         'count_nc_authors',
         'count_unity_ids']
    ])
else:
    print("✅ All valid-format rows match correctly.")

❌ MISMATCH FOUND (excluding invalid nc_state_people format rows) : 11


,title,authors,nc_state_people,nc_authors,unity_ids,count_nc_state_people,count_nc_authors,count_unity_ids
23,Ultra-High Alignment of Polymer Semiconductor ...,"Schrickx, Harry M.; Sen, Pratik; Booth, Ronald...",Harry Schrickx (hschric); Pratik Sen (psen2); ...,"Schrickx, Harry M.; Sen, Pratik; Booth, Ronald...",hschric; psen2; rebooth; aaaltaqu; mbiliro; kg...,10,9,10
383,Bacterial biodegradation and bioconversion of ...,"Mathews, Stephanie L.; Pawlak, Joel; Grunden, ...",Stephanie Mathews (sklambet); Sachin S. Pawar ...,"Mathews, Stephanie L.; Grunden, Amy M.; Pawlak...",sklambet; sspawar; amgrunde; jjpawlak,4,3,4
1412,QnAs with Rodolphe Barrangou,"Nair, Prashant",Rodolphe Barrangou (rbarran),NaN,rbarran,1,0,1
1417,Profile of Rodolphe Barrangou,"Davis, Tinsley H.",Rodolphe Barrangou (rbarran),NaN,rbarran,1,0,1
1439,Advances in sweetpotato breeding from 1992 to ...,"Grüneberg, W. J.; Ma, D.; Mwanga, R. O. M.; Ca...",Craig Yencho (yencho),NaN,yencho,1,0,1
1995,Poster abstract: Cyborg-insect networks for ma...,"Dirafzoon, A.; Bethhauser, J.; Schornick, J.; ...",Alireza Dirafzoon (adirafz); Joseph Leo Bettha...,"Dirafzoon, A.; Schornick, J.; Cole, J.; Bozkur...",adirafz; jlbettha; jcschorn; jacole; aybozkur;...,6,5,6
2072,Smart connected canines: IoT design considerat...,"Majikes, J. J.; Mealin, S.; Rita, B.; Walker, ...",John Majikes (jjmajike); Sean Patrick Mealin (...,"Majikes, J. J.; Mealin, S.; Roberts, D. L.; Wa...",jjmajike; spmealin; dlrober4; rbrugar; kdwalke...,8,7,8
2387,Preharvest Food Safety,"Libraries, NC State University",Siddhartha Thakur (sthakur),NaN,sthakur,1,0,1
2719,Potential Corn Yield Losses from Weeds in Nort...,"Soltani, Nader; Dille, J. Anita; Burke, Ian C....",Lewis Raver Braswell (lrbraswe); Charles Cahoo...,"Everman, Wesley J.",lrbraswe; cwcahoon; acyork; dljorda2; rwseagro...,6,1,6
2773,Rootstock improves high-tunnel tomato water us...,"Suchoff, D.H.; Schultheis, J.R.; Kleinhenz, M....",Paige H. Petticrew (plherrin); Helen Kraus (ht...,"Suchoff, D.H.",plherrin; htkraus; dhsuchof,3,1,3


In [23]:
rows_to_drop_mask = (~papers['has_invalid_nc_state_people_format']) & (~papers['counts_match'])

dropped_rows = papers[rows_to_drop_mask].copy()
papers_filtered = papers[~rows_to_drop_mask].copy().reset_index(drop=True)

print(f"Rows dropped: {len(dropped_rows)}")
print(f"Rows remaining: {len(papers_filtered)}")

Rows dropped: 11
Rows remaining: 3148


In [24]:
# Drop QC helper columns from final CSV
qc_helper_cols = [
    'count_nc_state_people',
    'count_nc_authors',
    'count_unity_ids',
    'has_invalid_nc_state_people_format',
    'counts_match'
]
papers_filtered = papers_filtered.drop(columns=qc_helper_cols)

# Save filtered dataframe to CSV
papers_filtered.to_csv("papers_filtered.csv", index=False)
print(f"Saved papers_filtered.csv with {len(papers_filtered)} rows and {papers_filtered.shape[1]} columns")


Saved papers_filtered.csv with 3148 rows and 9 columns
